In [100]:
import shap
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pandas as pd
from time import perf_counter
import numpy as np
import plotly.express as px

### Load arbitrary data

In [3]:
data = pd.read_parquet("../data/california_housing_prices.parquet")
X = data.drop(columns=["HousePrice"])
y = data["HousePrice"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Fit Linear ML Model

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

### Load shap linear implementation code

In [116]:
%run ../shap_implementation/linear_explainer.py

# **Independent Features**

In [117]:
X_background = X_train.sample(n=100, random_state=42)

## Linear Explainer for Independent Features

If we assume that all features in our data are independent of each other, meaning they do not correlate with one another, there is a substantially faster way of calculating exact SHAP values for linear models.

Instead of evaluating all $2^p$ possible feature subsets, we can utilize the coefficients of the linear model to compute the exact SHAP values in just $p$ iterations for each sample. This represents a substantial speedup while producing the same results, as will be proven later.

For a single feature $i \in N$, where $N$ is the set of all features, we calculate its marginal contribution as

$$
\phi_i(v) = \beta_i (x_i - \bar{X}_i)
$$

where $\beta_i$ is the model's coefficient for that feature, $x_i$ is the feature's value for the given sample and $\bar{X}_i$ is the mean value of that feature in the background dataset.

Hence, we can express the model's prediction $f(x)$ as

$$
f(x) = E[f(X)] + \sum_{i=1}^n \beta_i (x_i - \bar{X}_i)
= E[f(X)] + \sum_{i=1}^n \phi_i(v)
$$

with $n = |N|$.

This calculation can be performed in $0.31$s, whereas the same calculation on the same data using the exact explainer from the SHAP library takes $11.5$s directly.


In [118]:
start_t = perf_counter()

linear_explainer = IndependentLinearExplainer(model, X_background)
shap_values_linear = linear_explainer.explain(X_test)


print(f"Linear Explainer computation duration: {perf_counter() - start_t:.3f}s")

Linear Explainer computation duration: 0.310s


In [119]:
start_t = perf_counter()

exact_explainer = shap.ExactExplainer(model.predict, X_background)
shap_values_exact = exact_explainer(X_test)

print(f"Linear Explainer computation duration: {perf_counter() - start_t:.3f}s")

ExactExplainer explainer: 4913it [00:11, 44.68it/s]                           

Linear Explainer computation duration: 11.531s


### Linear computed SHAP values equal exact SHAP values

We can show this mathematically and prove it algorithmically afterwards.

We already know from the exact explainer that

$$
\phi_i(v) = \sum_{S \subseteq N\setminus \{i\}} \frac{|S|! (n - |S| - 1)!}{n!} [v(S \cup \{i\}) - v(S)]
$$

with $N$ being the set of all feautes and $n = |N|$.

For $v(S)$ we substitute all features not in $S$ from our sample datapoint into our background dataset according to the interventional SHAP formulation

$$
v(S) = E[f(X) | \text{ do } X_s = x_s]
$$

The predicition of a linear model is written as

$$
f(X) = \beta_0 + \beta_1 X_1 + \dots + \beta_n X_n = \beta_0 + \sum_{j=1}^n \beta_j X_j
$$

We can split the sum into 2 groups with features in S and features not in S

$$
f(X) = \beta_0 + \sum_{j \in S} \beta_j X_j + \sum_{j \notin S} \beta_j X_j
$$

Therefore,

$$
v(S) = E[\beta_0 + \sum_{j \in S} \beta_j X_j + \sum_{j \notin S} \beta_j X_j | \text{ do } X_s = x_s]
$$

So for every feature $j \in S$ we do $X_j = x_j$

$$
\sum_{j \in S} \beta_j X_j \implies \sum_{j \in S} \beta_j x_j
$$

We get 

$$
v(S) = E[\beta_0 + \sum_{j \in S} \beta_j x_j + \sum_{j \notin S} \beta_j X_j | \text{ do } X_s = x_s]
$$

as $\beta_0$ and $\sum_{j \in S} \beta_j x_j$ are now constant with respect to the desired feature $i$ it follows that

$$
v(S) = \beta_0 + \sum_{j \in S} \beta_j x_j + E[\sum_{j \notin S} \beta_j X_j | \text{ do } X_s = x_s]
$$

Using the linearity of expectation we get

$$
v(S) = \beta_0 + \sum_{j \in S} \beta_j x_j + \sum_{j \notin S} E[\beta_j] E[X_j | \text{ do } X_s = x_s] = \beta_0 + \sum_{j \in S} \beta_j x_j + \sum_{j \notin S} \beta_j E[X_j | \text{ do } X_s = x_s]
$$

Now we have to evaluate the value of $E[X_j]$ for every $j \notin S$. 
Since for every coalition value we calculate the mean prediction on the background dataset substituting values from our given sample for all features in $S$. 

All other features remain the same. Since calculating the mean for the final prediction and calculating the mean for the feature values in between and summing afterwards leads to the same result for a linear model, we just take the mean feature value as expected value for $X_j$.

Let's assume $j \in \{1, 2, 3\}$, so 
$$
f(X) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3
$$

since $v(S) = E[f(X) | \text{ do } X_s = x_s]$, we get

$$
E[f(X)] = \beta_0 + \beta_1 E[X_1] + \beta_2 E[X_2] + \beta_3 E[X_3]
$$

for an arbitrary subset $S = \{1, 3\}$ we get

$$
v(S) = \beta_0 + \beta_1 x_1 + \beta_2 E[X_2] + \beta_3 x_3
$$

Let M be the size of our background data, then

$$
v(S) = \frac{1}{M} \sum_{k=1}^M (\beta_0 + \beta_1 x_1 + \beta_2 X_2 + \beta_3 x_3) = \beta_0 + \frac{1}{M} \sum_{k=1}^M \beta_1 x_1 + \frac{1}{M} \sum_{k=1}^M \beta_2 X_2 + \frac{1}{M} \sum_{k=1}^M \beta_3 x_3
$$

Hence, with respect to the empirical background distribution, $E[X_2]=\bar X_2$ and analogously $E[X_j]=\bar X_j$ for every feature $j\notin S$.

Coming back to
$$
v(S) = \beta_0 + \sum_{j \in S} \beta_j x_j + \sum_{j \notin S} \beta_j E[X_j | \text{ do } X_s = x_s]
$$

we can now substitute $E[X_j]$
$$
v(S) = \beta_0 + \sum_{j \in S} \beta_j x_j + \sum_{j \notin S} \beta_j \bar{X_j}
$$

As we want to calculate the marginal contribution of feature $i$, we can explicitly pull it from the second sum. We differentiate between two cases: $S \cup \{i\}$ and just $S$.

If we union $i$ into $S$, we intervene on $X_i$ and set $X_i=x_i$.
$$
v(S \cup \{i\}) = \beta_0 + \sum_{j \in S} \beta_j x_j + \beta_i x_i + \sum_{j \notin S, j \neq i} \beta_j \bar{X_j}
$$
otherwise, we use its expected value $E[X_i]$ for $i \notin S$ and take its mean
$$
v(S) = \beta_0 + \sum_{j \in S} \beta_j x_j + \beta_i \bar{X_i} + \sum_{j \notin S, j \neq i} \beta_j \bar{X_j}
$$

To get the marginal contribution according to the exact shapley values formula, we must substract

$$
v(S \cup \{i\}) - v(S) = \beta_0 + \sum_{j \in S} \beta_j x_j + \beta_i x_i + \sum_{j \notin S, j \neq i} \beta_j \bar{X_j} - [\beta_0 + \sum_{j \in S} \beta_j x_j + \beta_i \bar{X_i} + \sum_{j \notin S, j \neq i} \beta_j \bar{X_j}]
$$

Every term expect for our feature $i$ cancels and we end up with
$$
v(S \cup \{i\}) - v(S) = \beta_i x_i - \beta_i \bar{X_i} = \beta_i (x_i - \bar{X_i})
$$

which is exactly the states formula for linear computation of shapley values found above. As this expression no longer depends on $S$, we can pull it out of our original sum

$$
\phi_i(v) = \beta_i (x_i - \bar{X_i}) \sum_{S \subseteq N\setminus \{i\}} \frac{|S|! (n - |S| - 1)!}{n!}
$$

as the shapley weights sum to 1, we are left with

$$
\phi_i(v) = \beta_i (x_i - \bar{X_i})
$$


Programmatically, we can verify this by comparing each SHAP value produced by our linear explainer with the corresponding value from the exact explainer.

Since floating-point arithmetic can introduce small numerical rounding errors, we use NumPy's `.isclose()` function to determine whether the values are sufficiently close:

$$
|a-b| \leq atol + rtol \times |b|
$$

We use the exact SHAP values as the reference values ($b$) and allow a relative error of up to $0.001%$ of the reference value.

The comparison confirms that all SHAP values match within the specified tolerance. The maximum absolute difference between the two explainers is only $3.31 \times 10^{-7}$.


In [124]:
matches = np.isclose(shap_values_linear, shap_values_exact.values, rtol=1e-05, atol=1e-08)

print(f"Linear SHAP matches Exact SHAP: {matches.all()}")
print(f"{matches.sum()} / {matches.size} values match")
print(f"Maximum difference: {np.abs(shap_values_linear - shap_values_exact.values).max():.2e}")

Linear SHAP matches Exact SHAP: True
39296 / 39296 values match
Maximum difference: 3.31e-07


## Baselines and additivity property

As stated above, 
$$
f(x) = E[f(X)] + \sum_{i=1}^n \beta_i (x_i - \bar{X_i})
$$

hence we need to compute $E[f(X)]$ to verify that the linear explainer fulfills the additivity property.

For the exact explainer, $E[f(X)]$ is computed as the mean model prediction over the original background dataset.

Let $M$ be the size of the background data distribution of a linear model:

$$
E[f(X)] = \frac{1}{M} \sum_{k=1}^M [\beta_0 + \sum_{i=1}^n \beta_i x_i^{(k)}]
$$

We can also rewrite this as
$$
E[f(X)] = \frac{1}{M} \sum_{k=1}^M [\beta_0 + \beta_1 x_1^{(k)} + \dots + \beta_n x_n^{(k)}] = \beta_0 + [\frac{1}{M} \sum_{k=1}^M \beta_1 x_1^{(k)} + \dots + \frac{1}{M} \sum_{k=1}^M \beta_n x_n^{(k)}] = \beta_0 + \beta_1 \frac{1}{M} \sum_{k=1}^M x_1^{(k)} + \dots + \beta_n \frac{1}{M} \sum_{k=1}^M  x_n^{(k)}
$$

Therefore, taking the final mean over all predictions is the same as summing all individual mean feature values plus the intercept.

$$
E[f(X)] = \beta_0 + \beta_1 \bar{X_1} + \dots + \beta_n \bar{X_n} = \beta_0 + \sum_{i=1}^n \beta_i \bar{X_i}
$$

Finally, 
$$
\phi_i(v) = \beta_i (x_i - \bar{X_i})
$$

$$
\sum_{i=1}^n \phi_i(v) + E[f(X)] = \sum_{i=1}^n \beta_i (x_i - \bar{X_i}) + \beta_0 + \sum_{i=1}^n \beta_i \bar{X_i} = \beta_0 + \sum_{i=1}^n \beta_i (x_i - \bar{X_i}) + \beta_i \bar{X_i}
$$

The mean terms cancel out and we get the definition of a linear model

$$
= \beta_0 + \sum_{i=1}^n \beta_i x_i - \beta_i \bar{X_i} + \beta_i \bar{X_i} = \beta_0 + \sum_{i=1}^n \beta_i x_i = f(x)
$$


Programmatically, we find that the maximum absolute error between the model prediction reconstructed from the SHAP values and the original model prediction $y_{\mathrm{pred}}$ is only $9.09 \times 10^{-13}$. We can therefore conclude that the additivity property is fulfilled to numerical precision, with the negligible residual error attributable to floating-point arithmetic and rounding errors.


In [125]:
shap_final_pred = linear_explainer.expected_value + np.sum(shap_values_linear, axis=1)
shap_pred_diff = abs(shap_final_pred - y_pred)

print(f"Max error:  {shap_pred_diff.max():.2e}")
print(f"Mean error: {shap_pred_diff.mean():.2e}")
print(f"Additivity: {np.allclose(shap_final_pred, y_pred)}")

Max error:  9.09e-13
Mean error: 3.13e-13
Additivity: True


# Intercorrelated Features

In [110]:
corr = X.corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    labels={"x": "Feature", "y": "Feature", "color": "Correlation"},
)

fig.show()